In [1]:
# Load Packages
from datetime import date, datetime, timedelta
import calendar
import pandas as pd
import numpy as np
import itertools
import copy
from numpy.linalg import multi_dot
from scipy.stats import norm
import matplotlib.pyplot as plt
import math

In [47]:

#The following function finds the date corresponding to the last thursday of a month in a year
#Input: Year, month - both integers
#Output: Python date of last thursday of the month
def find_last_thurs_date_of_month(year, month):
    calendar_of_the_month = calendar.monthcalendar(year, month)
    no_of_weeks = len(calendar_of_the_month)
    if (calendar_of_the_month[no_of_weeks-1][3] != 0):
        return date(year, month, calendar_of_the_month[no_of_weeks-1][3])
    else:
        return date(year, month, calendar_of_the_month[no_of_weeks-2][3])

#Function for getting previous weekday of a date
#Input: any python date
#Output: Previous weekday python date
def prev_weekday(date):
    if (date.weekday() == 0):
        return date - timedelta(days=3)
    elif (date.weekday() == 6):
        return date - timedelta(days=2)
    else:
        return date - timedelta(days=1)
    
#0-Mon, 1-Tues, 2-Wed, 3-Thurs, 4-Fri, 5-Sat, 6-Sun
def next_weekday(date):
    if (date.weekday() == 4):
        return date + timedelta(days=3)
    elif (date.weekday() == 5):
        return date + timedelta(days=2)
    else:
        return date + timedelta(days=1)   

#Function to Check for holidays- if holiday, the closing/expiry day is the previous working_day
#if holiday, select next working day
#Input: python date, holiday python list
#Output: Previous working python 
def prev_workday_if_holiday(date, holidays_list):
    if (date in holidays_list):
        return prev_workday_if_holiday(prev_weekday(date), holidays_list)
    else:
        return date

def next_workday(date, holidays_list):
    if (next_weekday(date) in holidays_list):
        return next_workday(next_weekday(date), holidays_list)
    else:
        return next_weekday(date)    
    
#The following function finds the date corresponding to the previous thursdays
#Input: Date - Python date format, No.of. thursdays back
#Output: The interested thursday python date
def date_of_prev_thurs(date, no_of_thurs_back):
    if (date.weekday() > 3):
        minus_days = date.weekday() - 3
        prev_thurs = date - timedelta(days=minus_days) -timedelta(days = 7 * (no_of_thurs_back-1))
        return prev_thurs
    else:
        prev_thurs = date - timedelta(days=no_of_thurs_back * 7)
        add_days = 3-prev_thurs.weekday()
        return (prev_thurs + timedelta(days=add_days))
    

#Load dataframe from csv file of the dats for that month:
#Input:
#     Expiry dates list of options to be hedged in python date format
#     prod_types of the expiries = "CE", "PE", "FUT"
#Output:
#      dataframe with keys as same name as csv file
def load_all_mthly_data(monthly_expiries_list, prod_type_lists=["FUT", "CE", "PE"], relative_path = "mkt_data_covid_region/"):
    df_dict = dict()
    for expiry_prod in list(itertools.product(monthly_expiries_list, prod_type_lists)):
        start_date = date_of_prev_thurs(expiry_prod[0], no_of_thurs_back=4)
        if (expiry_prod[1] == "FUT"):
            file_name = expiry_prod[1] + "IDX_BANKNIFTY_" \
                        + start_date.strftime("%d-%b-%Y") +  "_TO_" \
                        + expiry_prod[0].strftime("%d-%b-%Y") + ".csv"
        else:
            file_name = "OPTIDX_BANKNIFTY_" + expiry_prod[1] + "_" \
                        + start_date.strftime("%d-%b-%Y") +  "_TO_" \
                        + expiry_prod[0].strftime("%d-%b-%Y") + ".csv"
        df_dict["df" + "_" + file_name[:-4]] = pd.read_csv(relative_path + file_name)
    return (df_dict)


# The function helps in finding the list option last thursday expiries of the month 
#(we are interest to hedge options of this expiry)
# Input: Dictionary with keys as years and each key has value of list of months we are interested
# Output: List of monthly last thursday expiries
def get_all_monthly_option_expiries(periods_of_interest, holidays_list):
    monthly_expiries = []
    for key in periods_of_interest.keys():
        list_of_months = periods_of_interest[key]
        expiry_dates = [find_last_thurs_date_of_month(key, month) for month in list_of_months]
        expiry_dates = [prev_workday_if_holiday(date, holidays_list) for date in expiry_dates]
        monthly_expiries = monthly_expiries + expiry_dates
    return monthly_expiries

#Function to find all weekly expiry option dates for every month of interest (i.e. every thursdays)
#Input: List python monthly expiries
#Output: dictionary - {'monthly_expiry_date:': [4weeks before thurs, 3w, 2w, 1w]}
def find_wkly_expries(mthly_expiries, holidays_list):
    weekly_expiries_dict = {}
    for expiry_date in mthly_expiries:
        wkly_expiries_list = [date_of_prev_thurs(expiry_date, no_of_thurs) for no_of_thurs in range(4, 0, -1)]
        wkly_expiries_list = [prev_workday_if_holiday(date, holidays_list) for date in wkly_expiries_list]
        weekly_expiries_dict[expiry_date.strftime("%d-%b-%Y")] = wkly_expiries_list
    return weekly_expiries_dict

def generate_daily_dates_each_month(dict_wkly_expiries_each_month, holidays_list):
    dict_daily_dates = {}
    for each_month in dict_wkly_expiries_each_month:
        start_date = dict_wkly_expiries_each_month[each_month][0]
        end_date = datetime.strptime(each_month, "%d-%b-%Y").date()
        curr_date = start_date
        monthly_list = []
        while (curr_date < end_date):
            curr_date = next_workday(curr_date, holidays_list)
            monthly_list.append(curr_date)
        dict_daily_dates[each_month] = monthly_list
    return dict_daily_dates

# Funtion to get monthly data_frame names stored for each mkt_data from expiry_date
#Input: date, product type is "CE", "PE" or "FUT"
#Output: dataframe string name
def get_mthly_df_name_from_expiry(expiry_date, prod_type, holidays_list):
    start_date = date_of_prev_thurs(expiry_date, no_of_thurs_back=4)
    start_date = prev_workday_if_holiday(start_date, holidays_list)
    if (prod_type == "FUT"):
        df_name_str = "df_" + prod_type + "IDX_BANKNIFTY_" \
                    + start_date.strftime("%d-%b-%Y") +  "_TO_" \
                    + expiry_date.strftime("%d-%b-%Y")
    else:
            df_name_str = "df_" + "OPTIDX_BANKNIFTY_" + prod_type + "_" \
                        + start_date.strftime("%d-%b-%Y") +  "_TO_" \
                        + expiry_date.strftime("%d-%b-%Y")      
    return df_name_str


#Function for interpolation of iv - Linear Interpolation
#Other standard ways of interpolation could be tried in future
def gen_interpolated_iv(df_iv, vol_date_str, moneyness):
#     vol_date_str = df_for_lambda['Date']
#     moneyness = df_for_lambda['Moneyness']
    df_surf = df_iv[df_iv['Date']==vol_date_str]
    k_by_s = np.array(df_surf['Moneyness'])
    imp_vol = np.array(df_surf['Impl_Vol'])
    if (moneyness <= k_by_s[0]):
        return imp_vol[0]
    elif (moneyness >= k_by_s[-1]):
        return imp_vol[-1]
    curr_moneyness = k_by_s[0]
    ind = 0
    while (moneyness >= curr_moneyness):
        ind = ind + 1
        curr_moneyness = k_by_s[ind]
    interpolated_value = imp_vol[ind-1] + ((imp_vol[ind] - imp_vol[ind-1]) / (k_by_s[ind] - k_by_s[ind-1])) * ((moneyness - k_by_s[ind-1]))
    return interpolated_value

#Delta Hedging by Futures
def call_delta_with_futures(s_t, k, r, vol, dt):
    d_1_c = (1 / (vol * (dt ** 0.5))) 
    d1_log =  (np.log((s_t)/(k)) + ((r + (vol ** 2) / 2) * dt))
    d_1 = d_1_c * d1_log
    dc_ds = norm.cdf(d_1)
    df_ds = np.exp(r * dt)
    dc_df = float(dc_ds/df_ds)
    return dc_df

# def put_delta_with_futures(s_t, k, r, vol, dt):
#     d_1_c = (1 / (vol * (dt ** 0.5))) 
#     d1_log =  (np.log((s_t)/(k)) + ((r + (vol ** 2) / 2) * dt))
#     d_1 = d_1_c * d1_log
#     dc_ds = norm.cdf(d_1)
#     df_ds = np.exp(r * dt)
#     dc_df = float(dc_ds/df_ds)
#     return dc_df

def dynamic_hedge_pfl_error(mthly_expiries_list, mthly_mkt_data,\
                           prod_type, prod_moneyness, holidays_list, \
                           relative_path = "mkt_data_covid_region/"):
    
    r = 0.03 #To be changed after discussing with sir
    #On similar lines, futre price is now S * e^rt - to be taken real data once discussion on r is done
    mth_list_of_df = []
    df_strikes_hedged = pd.read_csv(relative_path + "code_outputs/B0001_" + prod_type + "_strikes_hedged.csv")

    for each_month in mthly_expiries_list:
        strike = df_strikes_hedged[df_strikes_hedged['Date'] == each_month.strftime("%d-%b-%Y")]['Strike'].iloc[0]
        df_month = mthly_mkt_data[get_mthly_df_name_from_expiry(each_month, prod_type, holidays_list)]
        df_month = df_month[['Date', 'Expiry', 'Underlying Value', 'Strike Price', 'Close']]
        df_month = df_month[df_month['Strike Price'] == strike]
        
        if (prod_type == "CE"):
            df_month["Moneyness"] = np.divide(df_month['Underlying Value'], df_month['Strike Price'])
        else:
            df_month["Moneyness"] = np.divide(df_month['Strike Price'], df_month['Underlying Value'])
        
        df_daily_iv_data = pd.read_csv(relative_path + "code_outputs/" + "Daily_vol_smile_CE_" + each_month.strftime("%d-%b-%Y")  + ".csv")
        
        if (prod_type == "PE"):
            df_iv = copy.deepcopy(df_daily_iv_data)
            df_iv['Moneyness'] = np.reciprocal(df_daily_iv_data['Moneyness'])
        else:
            df_iv = copy.deepcopy(df_daily_iv_data)
        
        df_month_to_find_delta = df_month[df_month['Date'] != each_month.strftime("%d-%b-%Y")]
        
#         print(df_iv.head())
#         df_month['iv'] = df_month.apply(gen_interpolated_iv(df_iv), axis=1)
        df_month_to_find_delta['iv'] = df_month_to_find_delta.apply(lambda x: gen_interpolated_iv(df_iv, x['Date'], x['Moneyness']), axis=1)
        df_month_to_find_delta['dt'] = df_month_to_find_delta.apply(lambda x: float((datetime.strptime(x['Date'], "%d-%b-%Y").date()-  each_month).days) / 365, axis=1)
        df_month_to_find_delta['delta'] = df_month_to_find_delta.apply(lambda x: \
                                          call_delta_with_futures(x['Underlying Value'], x['Strike Price'], \
                                                                  r, x['iv'], x['dt']), axis=1)
        df_month_to_find_delta['Futures'] = df_month_to_find_delta.apply(lambda x: x['Underlying Value'] * np.exp(r * x['dt']), axis=1)

        
        df_month_hedge_error = df_month[df_month['Date'] != df_month['Date'].iloc[0]]
        df_month_hedge_error['dt'] = df_month_hedge_error.apply(lambda x: float((datetime.strptime(x['Date'], "%d-%b-%Y").date()-  each_month).days) / 365, axis=1)
        df_month_hedge_error['Futures'] = df_month_hedge_error.apply(lambda x: x['Underlying Value'] * np.exp(r * x['dt']), axis=1)
        df_month_hedge_error['delta'] = df_month_to_find_delta['delta']
        df_month_hedge_error['Futures_lag'] = np.array(df_month_to_find_delta['Futures'])
        df_month_hedge_error['Close_lag'] = np.array(df_month_to_find_delta['Close'])
        df_month_hedge_error['price_change'] = df_month_hedge_error['Close'] - df_month_hedge_error['Close_lag']
        df_month_hedge_error['futures_change'] = df_month_hedge_error['Futures'] - df_month_hedge_error['Futures_lag'] 
        
        df_month_hedge_error['Dynamic Hedge Error']  = (df_month_hedge_error['price_change']) - (df_month_to_find_delta['delta'] * df_month_hedge_error['futures_change'] )
        df_month_hedge_error = df_month_hedge_error[['Date', 'Close', 'Dynamic Hedge Error']]

        mth_list_of_df.append(df_month_hedge_error) 
        
    df_dynamic_hedge = pd.concat(mth_list_of_df, axis=0)   
    df_dynamic_hedge.to_csv(relative_path + "code_outputs/D0001_" + prod_type + "_" + prod_moneyness + "_"+ "Dynamic_delta_hedge_error.csv", index = False)

    return df_dynamic_hedge


In [48]:
#Compute Daily Dynamic Hedging Errors

###################################
# Common Variables initialisation #
###################################

#Path to refer data
source_path = "mkt_data_covid_region/"

#Static hedging performed at different moneyness regions 
#i.e. moneyness is used to select the option with nearest moneyness match
# ATM - At the Money, ITM - In the money, OTM - Out of the Money
prod_moneyness = "ATM"

#Product type to hedge: either "CE" or "PE"
prod_type = "CE"

# Periods of interest will be a dictionary
#Key is the year, value is a list of months 1-12, 1- Jan, 2 - Feb,...12 - Dec
# For E.g., periods_of_interest = {2020: [3], 2019: [11, 12]}
periods_of_interest = {2020: [3]}

#List of holidays
holidays_list = [date(2020, 3, 10)]


#Find the monthly strike of option from mkt data to find the option to be hedged
#Every month, an option is hedged
mthly_expiries_list = get_all_monthly_option_expiries(periods_of_interest, holidays_list)
dict_wkly_expiries_each_month = find_wkly_expries(mthly_expiries_list, holidays_list)
dict_daily_dates_error_cal = generate_daily_dates_each_month(dict_wkly_expiries_each_month, holidays_list)


#Load all monthly mkt data
mthly_mkt_data = load_all_mthly_data(mthly_expiries_list, prod_type_lists=["FUT", "CE", "PE"], relative_path = source_path)

df_dynamic_hedge = dynamic_hedge_pfl_error(mthly_expiries_list, mthly_mkt_data,\
                                           prod_type, prod_moneyness, holidays_list, \
                                           relative_path = source_path)


/opt/tljh/user/lib/python3.6/site-packages/ipykernel_launcher.py:209: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/opt/tljh/user/lib/python3.6/site-packages/ipykernel_launcher.py:210: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/opt/tljh/user/lib/python3.6/site-packages/ipykernel_launcher.py:165: ComplexWarning: Casting complex values to real discards the imaginary part
/opt/tljh/user/lib/python3.6/site-packages/ipykernel_launcher.py:213: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sli